# Notebook 01 — K₃ QAOA demo

Step-by-step walkthrough of the QAOA pipeline on the triangle graph K₃ at p=1, following brief Section 3.2 (Module 7).  Each cell corresponds to one stage of the pipeline.

## Setup

In [ ]:
import os, sys
from pathlib import Path
# Move CWD to the project root so relative paths (configs/, results/) work
# regardless of where the notebook is launched from.
_here = Path.cwd()
if _here.name == 'notebooks':
    os.chdir(_here.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print('cwd:', Path.cwd())

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from src import visualizations as viz
from src.graph_generator import generate_triangle
from src.hamiltonian import build_cost_hamiltonian, optimal_cut_value
from src.circuit import qaoa_circuit
from src.metrics import approximation_ratio

viz.setup_style()

## Cell 1 — Generate K₃, show graph

In [ ]:
G = generate_triangle()
print(f'nodes: {list(G.nodes())}')
print(f'edges: {list(G.edges())}')
print(f'optimal cut value C*: {optimal_cut_value(G)}')

fig = viz.make_figure_09(n_3reg=6, seed_3reg=0)
plt.show()

## Cell 2 — Build H_C, print spectrum

With Option 1 convention, eigenvalues of H_C are **-C(z)** — ground-state energy −C\* = −2 for K₃.

In [ ]:
from qiskit.quantum_info import Statevector

H = build_cost_hamiltonian(G)
print('H_C:')
print(H)

eigs = np.linalg.eigvalsh(H.to_matrix())
print(f'\neigenvalues: {sorted(eigs.round(6).tolist())}')
print(f'ground-state energy: {eigs.min():.4f}  (should be -C* = -2)')

## Cell 3 — Build circuit, visualize

In [ ]:
# Use the K3 detailed run to get optimal (γ*, β*) — picks up the cached objects.
k3 = viz.collect_k3_detailed(grid_size=20, shots=5_000)
print(f'γ* = {k3.gamma_opt:.4f},  β* = {k3.beta_opt:.4f}')

fig = viz.make_figure_01(k3)
plt.show()

## Cell 4 — Optimize (grid + COBYLA), print parameters

The grid-search → COBYLA path is executed inside `collect_k3_detailed`. We print the final values and report the expectation.

In [ ]:
print(f'optimization finished after {len(k3.convergence)} COBYLA evaluations')
print(f'final ⟨H_C⟩ = {k3.final_expectation:.6f}  (optimum = -{k3.c_star})')
print(f'γ* = {k3.gamma_opt:.6f},  β* = {k3.beta_opt:.6f}')

## Cell 5 — Histogram, optimization curve, landscape

In [ ]:
fig2 = viz.make_figure_02(k3); plt.show()
fig3 = viz.make_figure_03(k3); plt.show()
fig4 = viz.make_figure_04(k3); plt.show()

## Cell 6 — Approximation ratio vs Farhi 2014 bound

The Farhi bound 0.6924 applies only to **3-regular** graphs (K₃ is 2-regular, so the bound isn't formally applicable — we print it for reference).

In [ ]:
print(f'approximation ratio r = {k3.approx_ratio:.6f}')
print(f'Farhi 2014 bound (3-regular p=1): r ≥ 0.6924')
print(f'K₃ p=1 saturates at r = 1.0 in this implementation')

# Sanity: uniform superposition baseline = 0.75 for K₃.
print(f'\nbaseline r for uniform superposition on K₃ = 0.75')